# 17 — Preliminary channel ladder (Modal batch smoke)

Priority-1 plumbing notebook: one ``gsin_upc_diag`` shard plus a short closed-loop
profit ladder (P0 / P1 / F2a / F3) with oracle reference.

Defaults target **Modal** via ``run_batch``; set ``BATCH_MODE = "local"`` for
ProcessPoolExecutor on a laptop (requires ``maturin develop`` + gsin binary).

**Before running:** from the repo root (not ``notebooks/``), build artifacts:

```bash
cargo build -p voi_core --release --example gsin_upc_diag
uv run maturin build --release -o dist/wheel
```

Restart the Jupyter kernel after building or changing env vars — Modal caches the
image definition at first import.

In [1]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Literal

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for _candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (_candidate / "src" / "blueberries_voi").is_dir():
        REPO_ROOT = _candidate
        break

_wheel_dir = REPO_ROOT / "dist" / "wheel"
_gsin_bin = REPO_ROOT / "target" / "release" / "examples" / "gsin_upc_diag"
os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")
if _wheel_dir.is_dir():
    os.environ["BLUEBERRIES_VOI_WHEEL"] = str(_wheel_dir)
if _gsin_bin.is_file():
    os.environ["GSIN_UPC_DIAG_BIN"] = str(_gsin_bin)

from blueberries_voi.experiments.modal_dispatch import run_batch
from blueberries_voi.filter.types import channels_cache_key, channels_for_preset

BATCH_MODE: Literal["modal", "local"] = "modal"
SMOKE = False  # full prelim grids below; set True to shrink further

GSIN_CELLS = [(2, 0)]  # heterogeneous deep-shelf regime, first seed
PROFIT_SEEDS = (42,)
PROFIT_PRESETS = ("P0", "P1", "F2a", "F3")
PROFIT_CHANNELS = [channels_for_preset(s) for s in PROFIT_PRESETS]
LABELS = {
    channels_cache_key(ch): s for s, ch in zip(PROFIT_PRESETS, PROFIT_CHANNELS, strict=True)
}
N_BURN, N_SCORE = 2, 5

## GSIN shard (one cell)

In [2]:
gsin_rows = run_batch(
    "gsin",
    BATCH_MODE,
    smoke=SMOKE,
    gsin_cells=GSIN_CELLS,
)
gsin_df = pd.DataFrame(gsin_rows)
gsin_df[["regime", "channel", "count_mae", "store_mean_f_mae"]]

[2026-08-24T17:05:01Z] nb13 grid: 1/1 complete


[2026-08-24T17:05:01Z] modal batch collected 1 shards


,regime,channel,count_mae,store_mean_f_mae
0,"Heterogeneous fleet, deep shelf",F1,202.4717,0.116217
1,"Heterogeneous fleet, deep shelf",F2,136.4630,0.072134
2,"Heterogeneous fleet, deep shelf",F2a,2.6778,0.023254
3,"Heterogeneous fleet, deep shelf",F3,1.2334,0.009527
4,"Heterogeneous fleet, deep shelf",P0,9.7006,0.088563
5,"Heterogeneous fleet, deep shelf",P1,5.2205,0.085136


## Closed-loop profit ladder + oracle

In [3]:
profit_rows = run_batch(
    "voi_profit",
    BATCH_MODE,
    smoke=SMOKE,
    seeds=PROFIT_SEEDS,
    channels=PROFIT_CHANNELS,
    include_oracle=True,
    n_burn=N_BURN,
    n_score=N_SCORE,
    n_rollout_paths=0,
)
profit_df = pd.DataFrame(profit_rows)
profit_df["label"] = profit_df["key"].map(LABELS).fillna(profit_df.get("preset", ""))
profit_df.groupby("label")[["profit", "waste", "stockout"]].mean().sort_values("profit")

[2026-08-24T17:05:12Z] nb13 grid: 1/5 complete


[2026-08-24T17:05:12Z] nb13 grid: 2/5 complete


[2026-08-24T17:05:13Z] nb13 grid: 3/5 complete


[2026-08-24T17:05:13Z] nb13 grid: 4/5 complete


[2026-08-24T17:05:15Z] nb13 grid: 5/5 complete


[2026-08-24T17:05:15Z] modal batch collected 5 shards


,profit,waste,stockout
label,,,
B-state,81.5,0.0,0.0
F2a,130.0,2.0,17.0
F3,130.0,2.0,17.0
P0,131.5,1.0,17.0
P1,131.5,1.0,17.0
